In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "0" 
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic" 
os.environ['MKL_THREADING_LAYER'] = "GNU"
import torch 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource
import time 

In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [5]:
if is_main:
    seed = 46
    environment_string = "pong"
    gold_timesteps = 15_000_000
    training_timesteps = 250_000
    num_concepts_selected = 11
    out_folder = "basic"
    method = "lp" 


In [6]:
gold_timesteps = {
    'cart_pole': 4_000_000,
    'mini_grid': 500_000,
    'pong': 15_000_000,
    'boxing': 30_000_000,
}

In [7]:
if is_main:
    concept_list, processed_concepts = get_concepts(environment_string,"human_selected_binary",seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env = get_environment(environment_string, None, seed)   
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps[environment_string],seed)
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [11]:
reward = evaluate_model(environment_string,ground_truth_gym_env,groundtruth_model,seed,max_steps=250_000)
reward 

True True 885.0
True True 1655.0
True True 1655.0
True True 1668.0
True True 1668.0
True True 1702.0
True True 1938.0
True True 2218.0
True True 1655.0
True True 885.0
True True 1655.0
True True 1656.0
True True 1685.0
True True 1655.0
True True 1829.0
True True 1655.0
True True 1961.0
True True 1628.0
True True 1668.0
True True 1728.0
True True 2633.0
True True 1792.0
True True 2480.0
True True 885.0
True True 1668.0
True True 2455.0
True True 1788.0
True True 1655.0
True True 2350.0
True True 1668.0
True True 1655.0
True True 1655.0
True True 2306.0
True True 1394.0
True True 2317.0
True True 1204.0
True True 1686.0
True True 2639.0
True True 1655.0
True True 1655.0
True True 1728.0
True True 1655.0
True True 1669.0
True True 1668.0
True True 2468.0
True True 1685.0
True True 1668.0
True True 1655.0
True True 1858.0
True True 1745.0
True True 2508.0
True True 1655.0
True True 1951.0
True True 1655.0
True True 1685.0
True True 2365.0
True True 1668.0
True True 824.0
True True 2136.0
T

10.242647058823529

In [70]:
model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps[environment_string],seed,"q_value","human_selected_binary")
if os.path.exists(model_name):
    q_estimates = pickle.load(open(model_name,"rb"))
else:
    q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
    pickle.dump(q_estimates,open(model_name,"wb"))


In [71]:
subset_concept, idx = policy_coverage_selection_lp_hybrid(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,
    q_estimates)

7


There are 80000 observations


In [72]:
predictor_epochs = 100
model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,predictor_epochs,seed)

height = width = 84

if environment_string == "mini_grid":
    num_frames = 1
else:
    num_frames = 4

if environment_string == "cart_pole":
    height = 160
    width = 240

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if os.path.exists(model_name):
    concept_predictor = ConceptPredictorCNN(len(concept_list), num_frames=num_frames,height=height,width=width).to(device)
    concept_predictor.load_state_dict(torch.load(model_name, weights_only=True))
    concept_predictor.eval()

In [95]:
intervention_prob = 0.5
two_stage_env, two_stage_gym_env = get_environment(environment_string,subset_concept,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,intervention_prob=intervention_prob,processed_concepts=processed_concepts)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [96]:
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_intervention_{}_{}_{}".format(environment_string,method,intervention_prob,seed))    
reward = evaluate_model(environment_string,two_stage_gym_env,model,seed)
reward

approx_kl,▁▁▁▁▁▂▂▁▁▁▁▁▁▁▁▃▃▂▂▂▂▂▂▂▂▂▂▂▃▃▄▄▅▅▄▅▅▅▅█
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▄▅█▆██
ema_norm_reward,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▂▃▃▅▅▄▅▅▅▅▆▆▇▅▆▆▆▇▇████
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▃▂▂▃▃▃▃▄▅▅▅▆▆▇▇▇▇█
episode_length_mean,▆██████▂███▆██▆██▅█▆█▇▂▅█▁▄▄▆▆▁▃▂▁▄▁▁▄▁▃
episode_reward_max,▁▁▁▁▁▁▁▁▁▆▁▆▄▁▁▁▅▁▁▁▅▁▄▄██▅█▇▇▁▆▆█▅▇██▇█
episode_reward_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▃▃▆▅▄▁▇▆▁▇▇▇▇▅▆██▇█
episode_reward_min,▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▇▆▁▃█▇▂▄▅▆██▇▇█▄▇▆█
episodes_completed,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇██
explained_variance,▁▁▆▆▅▅▅▅▅▄▅▇▇▆▆▆▆▅▅▆▇▇███▇▇▇▇▇▇▇▇██████▇
+1,...


Mask is starting tensor([[ True,  True, False, False, False,  True, False,  True,  True,  True,
         False],
        [ True,  True, False,  True, False,  True, False, False, False,  True,
          True],
        [ True, False,  True, False, False,  True,  True, False, False,  True,
          True],
        [False,  True, False, False,  True,  True, False,  True,  True, False,
          True],
        [False,  True,  True, False,  True,  True, False, False, False,  True,
          True],
        [ True, False,  True, False, False,  True, False,  True,  True,  True,
         False],
        [ True,  True, False,  True,  True,  True, False, False,  True, False,
         False],
        [ True, False,  True, False,  True,  True, False, False,  True,  True,
         False]], device='cuda:0')
Mask is ending tensor([[ True,  True, False, False, False,  True, False,  True,  True,  True,
         False],
        [ True,  True, False,  True, False,  True, False, False, False,  True,
       

0.844309052343542

In [89]:
reward 

Mask is starting tensor([[ True,  True, False, False, False,  True, False,  True,  True,  True,
         False],
        [ True,  True, False,  True, False,  True, False, False, False,  True,
          True],
        [ True, False,  True, False, False,  True,  True, False, False,  True,
          True],
        [False,  True, False, False,  True,  True, False,  True,  True, False,
          True],
        [False,  True,  True, False,  True,  True, False, False, False,  True,
          True],
        [ True, False,  True, False, False,  True, False,  True,  True,  True,
         False],
        [ True,  True, False,  True,  True,  True, False, False,  True, False,
         False],
        [ True, False,  True, False,  True,  True, False, False,  True,  True,
         False]], device='cuda:0')
Mask is ending tensor([[ True,  True, False, False, False,  True, False,  True,  True,  True,
         False],
        [ True,  True, False,  True, False,  True, False, False, False,  True,
       

0.7477413396986704

In [86]:
reward = evaluate_model(environment_string,two_stage_gym_env,model,seed)

Mask is starting tensor([[ True,  True, False, False, False,  True, False,  True,  True,  True,
         False],
        [ True,  True, False,  True, False,  True, False, False, False,  True,
          True],
        [ True, False,  True, False, False,  True,  True, False, False,  True,
          True],
        [False,  True, False, False,  True,  True, False,  True,  True, False,
          True],
        [False,  True,  True, False,  True,  True, False, False, False,  True,
          True],
        [ True, False,  True, False, False,  True, False,  True,  True,  True,
         False],
        [ True,  True, False,  True,  True,  True, False, False,  True, False,
         False],
        [ True, False,  True, False,  True,  True, False, False,  True,  True,
         False]], device='cuda:0')
Mask is ending tensor([[ True,  True, False, False, False,  True, False,  True,  True,  True,
         False],
        [ True,  True, False,  True, False,  True, False, False, False,  True,
       

In [87]:
reward

0.01194243158950995

In [62]:
intervention_prob = 0.5
two_stage_env, two_stage_gym_env = get_environment(environment_string,subset_concept,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,intervention_prob=intervention_prob,processed_concepts=processed_concepts)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [63]:
two_stage_env.reset()

Reset mask to tensor([[ True, False,  True],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False]], device='cuda:0')
Reset mask to tensor([[ True, False,  True],
        [False,  True,  True],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False]], device='cuda:0')
Reset mask to tensor([[ True, False,  True],
        [False,  True,  True],
        [ True, False,  True],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False],
        [False, False, False]], device='cuda:0')
Reset mask to tensor([[ True, False,  True],
        [False,  True,  True],
        [ True, False,  True],
        [ True, False,  True],
        [False, False,

array([[ 0.7291511 ,  0.7301635 , -0.57736444],
       [-0.4357636 , -1.2908026 , -0.57736444],
       [ 0.7291511 ,  0.78195333, -0.57736444],
       [-1.6049742 ,  0.78025115, -0.57736444],
       [-1.6049742 ,  0.7913314 , -0.57736444],
       [ 0.7291511 , -1.2908026 ,  1.7295635 ],
       [ 0.7291511 ,  0.7886319 , -0.57736444],
       [ 0.7291511 , -1.2908026 ,  1.73445   ]], dtype=float32)

In [64]:
for i in range(600):
    two_stage_env.step([1 for i in range(8)])

Reset mask to tensor([[ True, False,  True],
        [False,  True,  True],
        [ True, False,  True],
        [False,  True,  True],
        [ True, False,  True],
        [ True,  True, False],
        [ True, False,  True],
        [ True,  True, False]], device='cuda:0')
Reset mask to tensor([[ True, False,  True],
        [ True,  True, False],
        [ True, False,  True],
        [False,  True,  True],
        [ True, False,  True],
        [ True,  True, False],
        [ True, False,  True],
        [ True,  True, False]], device='cuda:0')
Reset mask to tensor([[ True, False,  True],
        [ True,  True, False],
        [ True, False,  True],
        [False,  True,  True],
        [False,  True,  True],
        [ True,  True, False],
        [ True, False,  True],
        [ True,  True, False]], device='cuda:0')
Reset mask to tensor([[ True, False,  True],
        [ True,  True, False],
        [ True, False,  True],
        [False,  True,  True],
        [False,  True,

In [65]:
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=1,custom_name="{}_intervention_{}_{}_{}".format(environment_string,method,intervention_prob,seed))    

ema_norm_reward,▁▂▄▄▄▅▆▆▅▆▆▇▇▇██▇▇█▇▇▆▆
episode_length_mean,▁▅▃▂▃▄▇▃▂▅▂▇▄▄▆▅▄▄█▄▁▁▅
episode_reward_max,▂▃▄▃▃▄▅▂▂▄▂█▃▅▇▄▄▂▅▄▁▂▇
episode_reward_mean,▁▅▃▂▃▄▇▃▂▅▂▇▄▄▆▅▄▄█▄▁▁▅
episode_reward_min,▂▅▃▂▃▂▆▂▁▂▂▂▃▂▂▄▂▅█▂▂▂▁
episodes_completed,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇██
ema_norm_reward,24.3677
episode_length_mean,29
episode_reward_max,58
episode_reward_mean,29
episode_reward_min,11


In [66]:
groundtruth_reward = evaluate_model(environment_string,two_stage_gym_env,model,seed)


Mask is starting tensor([[ True, False,  True],
        [False,  True,  True],
        [ True, False,  True],
        [ True, False,  True],
        [ True, False,  True],
        [ True,  True, False],
        [ True, False,  True],
        [ True,  True, False]], device='cuda:0')
Mask is ending tensor([[ True, False,  True],
        [False,  True,  True],
        [ True, False,  True],
        [ True, False,  True],
        [ True, False,  True],
        [ True,  True, False],
        [ True, False,  True],
        [ True,  True, False]], device='cuda:0')


In [20]:
model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,15_000_000,44)
if os.path.exists(model_name):
    groundtruth_model = PPO.load(model_name)
groundtruth_reward = evaluate_model(environment_string,ground_truth_gym_env,groundtruth_model,seed)
groundtruth_reward

[19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0, 19.0]


19.0

In [15]:
np.mean([21.0, 21.0, 21.0, 21.0, 19.0, 18.0, 17.0, -1.0, 19.0, 19.0, 19.0, 18.0, 14.0, 17.0, 18.0, 18.0, 21.0, 21.0, 18.0, 18.0, 18.0, 12.0, 21.0, 2.0, 19.0, 16.0, -14.0, 18.0, 19.0, 21.0, 21.0, 13.0, 21.0, 21.0, 18.0, 19.0, 19.0, 21.0, 11.0, 15.0, 21.0, 21.0, 21.0, 18.0, -6.0, 18.0, 21.0, 19.0, 10.0, 21.0])

16.64

In [8]:
groundtruth_model

NameError: name 'groundtruth_model' is not defined

In [14]:
acc_list = [0.75 for i in range(len(concept_list))]

In [15]:
policy_coverage_selection_multiple(ground_truth_gym_env,concept_list,num_concepts_selected,groundtruth_model,q_estimates,acc_list)

2


There are 80000 observations
2
There are 80000 observations
2
There are 80000 observations
2
There are 80000 observations
There are 3.0 x vals
Coverage 0.85395


([<function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
  <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
  <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>],
 [1, 7, 10])

In [16]:
policy_coverage_selection_multiple_log(ground_truth_gym_env,concept_list,num_concepts_selected,groundtruth_model,q_estimates,acc_list)

2


There are 80000 observations
2
There are 80000 observations
2
There are 80000 observations
2
There are 80000 observations
Y Vals mean 0.8404
Y 2 Vals maen 0.9999999690298964
There are 3.0 x vals
Coverage 0.8404


([<function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
  <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>,
  <function concept_abstraction.concept_bank.less_function.<locals>.f_less(obs)>],
 [7, 9, 11])

In [13]:
subset, idx_policy = policy_coverage_selection_lp_hybrid(ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,q_estimates,rollout_steps=10_000)

2


ALSA lib confmisc.c:767:(parse_card) cannot find card '0'
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_card_driver returned error: No such file or directory
ALSA lib confmisc.c:392:(snd_func_concat) error evaluating strings
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1246:(snd_func_refer) error evaluating name
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5220:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2642:(snd_pcm_open_noupdate) Unknown PCM default
ALSA lib confmisc.c:767:(parse_card) cannot find card '0'
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_card_driver returned error: No such file or directory
ALSA lib confmisc.c:392:(snd_func_concat) error evaluating strings
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_concat returned error: N

There are 80000 observations
Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17
There are 3.0 x vals
Coverage 0.8224


In [15]:
idx_policy

[4, 7, 10]

In [7]:
subset, idx_policy = policy_coverage_selection_lp(ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,rollout_steps=10_000)

There are 80000 observations
Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17
Coverage 0.99555


In [38]:
subset, idx_hybrid = policy_coverage_selection_lp_hybrid(ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,q_estimates,rollout_steps=10_000)

6
There are 80000 observations
There are 55.0 x vals
6
There are 80000 observations
There are 57.0 x vals
Coverage 0.99515


In [42]:
len(set(idx_hybrid).intersection(set(idx_policy))),len(set(idx_hybrid).intersection(set(idx)))

(39, 46)

In [17]:
for a in unique_actions:
    relevant_idx = np.where(actions == a)[0]
    print(len(relevant_idx))

3508
3508
3508
3508
3508
3508


In [92]:
num_pairs_lp=20_000
rollout_steps=10_000

unique_actions = list(set([int(i[1]) for i in q_estimates]))
actions = np.array([i[1] for i in q_estimates])

# Continuous
discretized_X = np.array([i[0] for i in q_estimates])

q_values = np.array([i[2] for i in q_estimates])

final_vals = []
num_actions = len(set([i[1] for i in q_estimates]))
print(num_actions)
seen = set() 
for a in unique_actions:
    print(a)
    relevant_idx = np.where(actions == a)[0]
    if len(relevant_idx) <= 1000:
        relevant_low = relevant_high = relevant_idx
    else:
        relevant_low = random.sample(relevant_idx.tolist(),1000)
        relevant_high = random.sample(relevant_idx.tolist(),1000)
    for low_idx in relevant_low:
        for high_idx in relevant_high:
            diff = abs(q_values[low_idx] - q_values[high_idx])
            # tuple of differing concept indices
            diffs = tuple(i for i, (l, h) in enumerate(zip(discretized_X[low_idx], discretized_X[high_idx])) if l != h)
            tup = (diff, diffs)
            if diffs not in seen and diffs != ():
                seen.add(diffs)
                final_vals.append(tup)
print(len(final_vals))
final_vals = sorted(final_vals,reverse=True)
# final_vals = final_vals[:250_000]


6
0
1
2
3
4
5
3597024


In [30]:

# --------------------------------------------------
# Collect observations / actions (same as before)
# --------------------------------------------------
all_observations = []
all_actions = []

obs, info = ground_truth_gym_env.reset()

for _ in range(rollout_steps):
    actions = groundtruth_model.predict(obs)[0]
    for j in range(len(actions)):
        all_observations.append([c(info[j]['observation']) for c in concept_list])
        all_actions.append(actions[j])
    obs, rew, t_1, t_2, info = ground_truth_gym_env.step(actions)

all_observations = np.asarray(all_observations, dtype=np.int8)
all_actions = np.asarray(all_actions)

N, K = all_observations.shape

print("There are {} observations".format(N))

# --------------------------------------------------
# Sample cross-action pairs
# --------------------------------------------------
idx_i = np.random.randint(low=0, high=N, size=5 * num_pairs_lp)
idx_j = np.random.randint(low=0, high=N, size=5 * num_pairs_lp)

valid = all_actions[idx_i] != all_actions[idx_j]
idx_i = idx_i[valid][:num_pairs_lp]
idx_j = idx_j[valid][:num_pairs_lp]

if len(idx_i) == 0:
    raise ValueError("No cross-action pairs sampled.")

M = len(idx_i)

disagreement = (all_observations[idx_i] != all_observations[idx_j]).astype(np.int8)


There are 80000 observations


In [32]:
model = gp.Model("max_coverage_lp")
model.Params.OutputFlag = 0

ub = 1.0
len_x_vals = 0
trials = 0

# x_d variables (concept selection)
x = model.addVars(K, lb=0.0, ub=1.0, vtype=GRB.BINARY, name="x")

# y_p variables (pair covered)
y = model.addVars(M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="y")

y_2 = model.addVars(len(final_vals), lb=0.0, ub=ub, vtype=GRB.BINARY, name="y_2")

for i, (_, elems) in enumerate(final_vals):
    if elems:  # make sure not empty
        model.addConstr(
            y_2[i] <= gp.quicksum(x[e] for e in elems),
            name=f"cover_{i}"
        )
    else:
        model.addConstr(y_2[i] == 0)  # cannot be covered

# Prefix constraints: enforce consecutive coverage
# y[i] <= y[i-1] for i>0
for i in range(1, len(final_vals)):
    model.addConstr(y_2[i] <= y_2[i-1], name=f"prefix_{i}")

for p in range(M):
    model.addConstr(
        y[p] <= gp.quicksum(disagreement[p, d] * x[d] for d in range(K)),
        name=f"cover_{p}",
    )

# Cardinality constraint
model.addConstr(
    gp.quicksum(x[d] for d in range(K)) <= num_concepts_selected,
    name="budget",
)

# Constraint: maximize covered pairs
model.addConstr(gp.quicksum(y[p] for p in range(M))/M >= 0.99)
model.setObjective(gp.quicksum(y_2[i] for i in range(len(final_vals))), GRB.MAXIMIZE)    
model.optimize()

In [33]:
x_vals = np.array([x[d].X for d in range(K)])
y_vals = np.array([y[p].X for p in range(M)])
len_x_vals = sum(x_vals)


print("There are {} x vals".format(len_x_vals))
idx = [i for i in range(len(x_vals)) if x_vals[i] > 0.5]


There are 52.0 x vals


In [36]:
len(set(idx).intersection(set(idx_policy)))

38

In [43]:
np.mean([np.sum(i*x_vals)>0 for i in disagreement] )

0.9959

In [47]:
fixed_idx = deepcopy(idx)

In [71]:
model = gp.Model("max_coverage_lp")
model.Params.OutputFlag = 0

ub = 1.0
len_x_vals = 0
trials = 0

# x_d variables (concept selection)
x = model.addVars(K, lb=0.0, ub=1.0, vtype=GRB.BINARY, name="x")

# y_p variables (pair covered)
y = model.addVars(M, lb=0.0, ub=1.0,vtype=GRB.CONTINUOUS, name="y")

# y_2 = model.addVars(len(final_vals), lb=0.0,ub=1.0, vtype=GRB.CONTINUOUS, name="y_2")

# for i, (_, elems) in enumerate(final_vals):
#     if elems:  # make sure not empty
#         model.addConstr(
#             y_2[i] <= gp.quicksum(x[e] for e in elems),
#             name=f"cover_{i}"
#         )
#     else:
#         model.addConstr(y_2[i] == 0)  # cannot be covered

# Prefix constraints: enforce consecutive coverage
# y[i] <= y[i-1] for i>0
# for i in range(1, len(final_vals)):
#     model.addConstr(y_2[i] <= y_2[i-1], name=f"prefix_{i}")

for p in range(M):
    model.addConstr(
        y[p] <= gp.quicksum(disagreement[p, d] * x[d] for d in range(K)),
        name=f"cover_{p}",
    )

# Cardinality constraint
model.addConstr(
    gp.quicksum(x[d] for d in range(K)) <= num_concepts_selected,
    name="budget",
)

# for i in fixed_idx:
#     model.addConstr(x[i] == 1)

# Constraint: maximize covered pairs
# model.setObjective(gp.quicksum(weights[i]*y_2[i] for i in range(len(final_vals))), GRB.MAXIMIZE)    
model.setObjective(gp.quicksum(y[p] for p in range(M)), GRB.MAXIMIZE)    
model.optimize()

In [57]:
weights = [i[0] for i in final_vals]

In [73]:
x_vals = np.array([x[d].X for d in range(K)])
idx_policy = [i for i in range(len(x_vals)) if x_vals[i] == 1]
y_vals = np.array([y[p].X for p in range(M)])
print("Coverage {}".format(np.mean(y_vals)))


Coverage 0.996


In [100]:
x_vals = np.zeros((228))
x_vals[idx] = 1
np.mean([np.sum(i*x_vals)>0 for i in disagreement] )

0.9959

In [104]:
idx.pop()

6

In [86]:
set([6, 18, 44, 56, 61, 65, 68, 69, 72, 73, 74, 77, 80, 81, 84, 85, 86, 89, 92, 93, 94, 96, 97, 98, 101, 104, 105, 106, 108, 109, 110, 113, 116, 117, 119, 120, 122, 125, 126, 127, 129, 134, 138, 139, 140, 141, 143, 151, 152, 153, 155, 163, 165, 167, 176, 183, 195])-set(idx)

{6, 18, 44, 65, 69, 74, 77, 122, 126, 140, 141, 152, 153, 155, 167, 176, 183}

In [101]:
set(idx_policy)-set(idx)

{54, 69, 74, 140, 141, 155, 162, 175, 189}

In [135]:
r = [i for i in final_vals if set(i[1]).intersection(idx_policy) == set()]

KeyboardInterrupt: 

In [133]:
idx.append(77)
idx.append(140)
idx.append(74)

AttributeError: 'int' object has no attribute 'append'

In [106]:
values = np.zeros((len(r),len(concept_list)))
for idx,i in enumerate(r):
    values[idx,i[1]] = 1

In [132]:
values[(values[:,77]+values[:,74]+values[:,140]) == 0].shape

(59, 228)

In [122]:
jf = np.mean(values,axis=0)
np.argsort(-jf)

array([ 77, 140,  74, 175, 141, 177, 118, 176,  69, 121, 153, 188, 208,
       148, 122, 172, 160, 220, 152, 100,  88,  57, 124, 112, 136, 184,
       196, 219, 114,  90,  44, 164, 191, 102, 207,  51,  27, 189,  39,
        64,  45,  76,  28,  52,  40, 130, 157, 158, 163, 129, 128, 127,
       137, 131, 161, 162, 159, 145, 142, 156, 155, 154, 144, 133, 134,
       139, 151, 150, 135, 149, 143, 147, 146, 132, 138,   0, 166, 201,
       202, 203, 204, 205, 206, 209, 210, 211, 212, 213, 214, 215, 216,
       217, 218, 221, 222, 223, 224, 225, 200, 165, 199, 197, 167, 168,
       169, 170, 171, 126, 174, 178, 179, 180, 181, 182, 183, 185, 186,
       187, 190, 192, 193, 194, 195, 198, 173, 113, 123,  29,  30,  31,
        32,  33,  34,  35,  36,  37,  38,  41,  42,  43,  46,  47,  48,
        49,  50,  53,  54,  55,  26,  25,  24,  23,   1,   2,   3,   4,
         5,   6,   7,   8,   9,  10,  56,  11,  13,  14,  15,  16,  17,
        18,  19,  20,  21,  22,  12,  58,  59,  60,  94,  95,  9

In [105]:
q = set(list(range(300)))
for i in r:
    q = q.intersection(set(i[1]))
q

set()

In [103]:
len(r)

106

In [85]:
len(set([6, 18, 44, 56, 61, 65, 68, 69, 72, 73, 74, 77, 80, 81, 84, 85, 86, 89, 92, 93, 94, 96, 97, 98, 101, 104, 105, 106, 108, 109, 110, 113, 116, 117, 119, 120, 122, 125, 126, 127, 129, 134, 138, 139, 140, 141, 143, 151, 152, 153, 155, 163, 165, 167, 176, 183, 195]).intersection(set(idx_policy)))

37

In [81]:
len(set(idx).intersection(set(idx_policy))),len(set(idx).intersection(set(idx_hybrid)))

(33, 46)

In [60]:
len(set(idx).intersection(set(idx_policy))),len(set(idx).intersection(set(idx_hybrid)))

(38, 46)

In [61]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,processed_concepts=processed_concepts,concept_idx=idx)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=15_000_000,custom_name="pong_test")  

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
wandb: Currently logged in as: naveenr (naveenr-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


KeyboardInterrupt: 

In [50]:
len(idx)

54

In [52]:
subset, idx_new = policy_coverage_selection_lp_hybrid(ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,
    q_estimates,rollout_steps=10_000)

6
There are 80000 observations
There are 51.0 x vals
6
There are 8000 observations
There are 57.0 x vals
Coverage 0.99815


In [57]:
len(set(idx_new).intersection(set(idx_original)))/len(idx_original)

0.8421052631578947

In [40]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Optional but recommended for determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
model_params = get_model(environment_string,"MlpPolicy",custom_name="cart_pole_extension")
model_params['env'] = environment_string
ground_truth_env.seed(seed)


name = "cart_pole_extension"
groundtruth_model.env = ground_truth_env

In [47]:
subset_concept, idx = policy_coverage_selection_lp_hybrid(ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,
    q_estimates)

2
There are 8000 observations
Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17
2
There are 8000 observations
2
There are 8000 observations
There are 3.0 x vals
Coverage 0.85545


In [48]:
idx 

[5, 6, 7]

In [64]:
evaluate_model(environment_string,ground_truth_gym_env,groundtruth_model,seed)

498.61

In [8]:
subset_concept, idx = random_selection(concept_list,num_concepts_selected)

In [9]:
training_timesteps = 100_000


In [13]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,subset_concept,seed,processed_concepts=processed_concepts,concept_idx=idx)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_perfect_{}_{}".format(environment_string,method,seed),seed=seed)    

approx_kl,▁▁▁▁▁▁▁▁▃▃▂▂▂▁▁▁▁▃▃▃▃▃▂▂▂▂▂▆▆▃▃▅▅▅▅▆▆▆▆█
clip_fraction,▁▁▁▁▁▁▁▁▁▁▄▄▄▁▁▁▁▁▁▁▁▁▁▁▁▆▆▁▅▃▃▃▃▃██████
ema_norm_reward,▂▁▁▂▃▂▂▄▃▂▂▂▂▂▁▃▃▃▃▂▂▂▃▂▂▂▂▂▅▆▃▅▇▆▅▇▆▇██
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅███
episode_length_mean,█▄███████▃█████████████████▁██████▄██▆▅█
episode_reward_max,▁▁▁▁▅▁▁▁▁▇▁▁▁▃▂▄▁▁▁▁▁▇▁▁▁▄▁▁▁▁▇▄▁▄▁▇▁▁█▁
episode_reward_mean,▁▁▁▁▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁█▁▁▁▁▂▁▁▄▁▁▃▂▃
episode_reward_min,▁▁▁▁▇▄▁▂▁▁▁▁▁▁▁▁▁▁▂▁▁▁▂▂▁▁▂▆▁▁▃▇▁▇█▁▁▂█▁
episodes_completed,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇████
explained_variance,▁▁▁███████▇▇▇▆▆▆▅▅▅▅▆▆▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅
+1,...


In [11]:
model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,100,seed)

height = width = 84

if environment_string == "mini_grid":
    num_frames = 1
else:
    num_frames = 4

if environment_string == "cart_pole":
    height = 160
    width = 240

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if os.path.exists(model_name):
    concept_predictor = ConceptPredictorCNN(len(concept_list), num_frames=num_frames,height=height,width=width).to(device)
    concept_predictor.load_state_dict(torch.load(model_name, weights_only=True))
    concept_predictor.eval()

In [12]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,processed_concepts=processed_concepts)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_imperfect_{}_{}".format(environment_string,method,seed)) 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,▅▅▅▅▂▇▇▇▇▇▅▅▅▅▁▁▁▄▄▄▄▅▅▅▃▃▃███▂▂▂▂▄▄▄▃▃▃
clip_fraction,▄▄▄▄▁▃▃▃▃▃▄▄▁▁▁▁▁▁▁▁▃▃▃▁██████▁▁▁▁▁▁▁▁▁▁
ema_norm_reward,▂▃▂▂▂▁▁▂▁▁▁▁▁▁▁▃▂▂▂▂▃▁▂▂▂▂▂▂▂▁▄▆▅▆▅▄▄▆▃█
entropy_loss,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇███
episode_length_mean,█▇███████████████████▇▇▁███▂█▄█▅▆██▅█▃▅▅
episode_reward_max,▁▁▁▁▁▁▂▁▁▁▁▁▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆▅▁▄▄▃▁▃▃▁█▁▁
episode_reward_mean,█▃▁▁▁▁▁▁▁▁▁▃▁▂▃▁▁▁▁▁▁▁▁▁▁▁▄▁▁▁▁▁▁▂▅▃█▄▁▁
episode_reward_min,▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▄▁▁▁▁▁▁▃▁▄▁▁▁▄▁▁▁▁▅
episodes_completed,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇██████
explained_variance,▁▁▁▁▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇█████████
+1,...
